# Inspect one session (neuro+behavior) 

#### Get the recording of one session

In [17]:
%load_ext autoreload
%autoreload 2

from src.io import load_binary_session
import spikeinterface.preprocessing as spr
import probeinterface as pi

subject = "wifi"
session = "Wifi_20210618"
recording = load_binary_session(subject, session)
sig = recording.get_traces(start_frame=0, end_frame=1000)
# Load the probe configuration from the JSON file and attach it to the recording object
probegroup = pi.read_probeinterface('../utils/probe_layout.json')
recording = recording.set_probegroup(probegroup)
recording_uV = spr.scale(recording, gain=1e6)
print(f"Num channels: {recording.get_num_channels()} - Duration: {recording.get_duration()/60} min - Sampling frequency: {recording.get_sampling_frequency()} Hz")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Num channels: 128 - Duration: 104.43181145833333 min - Sampling frequency: 32000.0 Hz


#### Visually inspect raw data

In [ ]:
%matplotlib widget
import spikeinterface.widgets as sw

# Load the raw recording
sw.plot_traces(recording_uV, time_range=(0, 5), mode='map', backend='ipywidgets', clim=(-20, 20))

#### Visualize the mean spectrum of that session

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# Extract 5 seconds of data to achieve a clean 1 Hz frequency resolution
fs = recording_uV.get_sampling_frequency()
end_frame_5s = int(5 * fs)
traces = recording_uV.get_traces(start_frame=0, end_frame=end_frame_5s)

# Compute Power Spectral Density using Welch's method 
# nperseg=int(fs) forces the window size to exactly 1 second (1 Hz resolution)
frequencies, psd = welch(traces, fs=fs, nperseg=int(fs), axis=0)

# Average the PSD across all 128 channels to emphasize global common-mode noise
mean_psd = np.mean(psd, axis=1)

# Plot the spectrum focusing on the [0, 150] Hz range
plt.figure(figsize=(10, 5))
plt.semilogy(frequencies, mean_psd, color='black', linewidth=1.2)

# Highlight fundamental power-line frequency and its first harmonic
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50 Hz')
plt.axvline(x=100, color='orange', linestyle=':', alpha=0.7, label='100 Hz')

plt.xlim(0, 150)
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'Power Spectral Density ($\mu V^2$/Hz)')
plt.title('Averaged PSD of Raw Signal')
plt.legend(frameon=False)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Inspect behavior

In [7]:
import pandas as pd
from src.config import RAW_DATA_DIR, EVENT_SUFFIXES
from src.io import inspect_behavior

# Esecuzione agile per una specifica sessione
inspect_behavior("Wifi", "Wifi_20210618")

=== Behavior Summary: Wifi | Wifi_20210618 ===

[GRASP]
  - Total grasps recorded: 84
  - Breakdown by Target and Hand:
    floor   L       56
            R       12
    hook    L       12
            R        4

---------------------------------------------

[WALK]
  - Total walks: 7
  - Total individual steps: 52
  - Mean steps per walk: 7.4
  - Mean walk duration: 4.62 s
  - Walks breakdown by Surface:
    - Floor: 4 walks
    - Structure: 3 walks
